In [ ]:
import polars as pl
from scipy.stats import norm, poisson
import numpy as np
import datetime
import tensorflow as tf
import plotly.graph_objects as go
from typing import Literal

In [ ]:
df = pl.read_csv("data/results.csv", null_values=["NA"])

In [ ]:
df

In [ ]:
allTeams = df["home_team"].unique().to_list() + df["away_team"].unique().to_list()
allTeams = np.sort(list(set(allTeams)))
defendingParams = np.random.normal(1.5,0.4, len(allTeams))
attackingParams = np.random.normal(1.5,0.4, len(allTeams))
dfParams = pl.DataFrame({
    "team": allTeams,
    "num": range(len(allTeams)),
    "defending": defendingParams,
    "attacking": attackingParams
})
dictNum = {}
for i, team in enumerate(allTeams):
    dictNum[str(team)] = i
numTeams = len(allTeams)



In [ ]:
def getMatchMatrice(
    dfMatch: pl.DataFrame, 
    scope: pl.Expr,
    teamNumerotation: dict[str, int]
    ) -> np.ndarray:
    numTeams = len(teamNumerotation)
    df = dfMatch.filter(scope)
    res = np.zeros((df.height*2, numTeams*2 + 3))
    for i, row in enumerate(df.iter_rows(named=True)):
        #home team score
        res[2*i, teamNumerotation[str(row["home_team"])]] = 1
        res[2*i, numTeams+teamNumerotation[str(row["away_team"])]] = 1
        res[2*i, -1] = row["home_score"]
        #away team score
        res[2*i + 1, numTeams+teamNumerotation[str(row["home_team"])]] = 1
        res[2*i + 1, teamNumerotation[str(row["away_team"])]] = 1
        res[2*i + 1, -1] = row["away_score"]

        if not row["neutral"]:
            res[2*i, -3] = 1
            res[2*i + 1, -2] = 1
    return res

In [ ]:
scope = pl.lit(True)
scope = scope & (pl.col("date").is_between(datetime.date(2025, 6, 1), datetime.date(2026, 6, 1)))
scope = scope & (pl.all_horizontal([pl.col(column).is_not_null() for column in df.columns]))

matchMatrice = getMatchMatrice(df, scope, dictNum)
matchMatrice


In [ ]:
X[:,-3:-2]

In [ ]:
tf.reduce_sum(tf.random.normal((2,1), mean=0, stddev=1)*X[:,-3:-1], axis=1, keepdims=True)

In [ ]:
class MyModel(tf.keras.Model):
    def __init__(
        self,
        numTeams: int
    ):
        super().__init__()
        self.numTeams = numTeams
        self.teamParams = self.add_weight(
            name="team_params",
            shape=(2 * numTeams, 1),
            initializer=tf.random_normal_initializer(),
            trainable=True
        )
        self.awayHomeMultiplier = self.add_weight(
            name="away_home_multiplier",
            shape=(1, 2),
            initializer=tf.random_normal_initializer(),
            trainable=True
        )


    def call(self, input):
        attackingParams = self.teamParams[:self.numTeams]
        defendingParams = self.teamParams[self.numTeams:2*self.numTeams]

        attackingOneHotEncoder = input[:,:self.numTeams]
        defendingOneHotEncoder = input[:,self.numTeams:2*self.numTeams]
        score = input[:,-1]
        awayHomeMultiplier = tf.reduce_sum(input[:, -3:-1] * self.awayHomeMultiplier, axis=1, keepdims=True)
        scoreExpanded = tf.expand_dims(score, axis=1)

        params = tf.exp(awayHomeMultiplier + tf.matmul(attackingOneHotEncoder, attackingParams) - tf.matmul(defendingOneHotEncoder, defendingParams))
        
        poissonLogLikelyHood = -params + tf.math.log(params) * scoreExpanded - tf.math.lgamma(scoreExpanded + 1)

        return tf.reduce_sum(poissonLogLikelyHood)

In [ ]:
X = tf.constant(matchMatrice, dtype=tf.float32)
model = MyModel(numTeams=numTeams)

In [ ]:

model = MyModel(numTeams=numTeams)
optimizer = tf.keras.optimizers.Adam()
X = tf.constant(matchMatrice, dtype=tf.float32)
numEpoch = 10000
lossHistory = []
for i in range(numEpoch):
    print(f"Epoch {i+1}/{numEpoch}", end="\r")
    with tf.GradientTape() as tape:
        pred = -model(X)
        loss = pred
    lossHistory.append(loss)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(y=lossHistory, mode='lines', name='Loss'))
fig.update_layout(title='Loss History', xaxis_title='Epoch', yaxis_title='Loss')
fig.show()

In [ ]:
attack = [np.exp(model.teamParams.numpy())[i][0] for i in range(numTeams)]
defend = [np.exp(model.teamParams.numpy())[numTeams+i][0] for i in range(numTeams)]

In [ ]:
dictAttack = {str(allTeams[i]): attack[i] for i in range(numTeams)}
dictDefend = {str(allTeams[i]): defend[i] for i in range(numTeams)}
dictAwayHome = {"home": np.exp(model.awayHomeMultiplier.numpy())[0][0], "away": np.exp(model.awayHomeMultiplier.numpy())[0][1]}

In [ ]:
from audioop import mul
def getStatsDf(
    teamA: str, 
    teamB: str, 
    maxScore: int, 
    teamHome : Literal["first", "second", "neutral"]
    ) -> pl.DataFrame:
    dfDict = {
        teamA: [],
        teamB: [],
        "probability": []
    }
    multiplierA, multiplierB = 1, 1
    if teamHome == "first":
        multiplierA = dictAwayHome["home"]
        multiplierB = dictAwayHome["away"]
    elif teamHome == "second":
        multiplierA = dictAwayHome["away"]
        multiplierB = dictAwayHome["home"]

    for i in range(maxScore + 1):
        for j in range(maxScore + 1):
            dfDict[teamA].append(i)
            dfDict[teamB].append(j)
            prob = 100*poisson.pmf(i, multiplierA * dictAttack[teamA] / dictDefend[teamB]) * poisson.pmf(j, multiplierB * dictAttack[teamB] / dictDefend[teamA])
            dfDict["probability"].append(prob)
    res= pl.DataFrame(dfDict).sort("probability", descending=True)
    teamAWins = pl.col(teamA) > pl.col(teamB)
    teamBWins = pl.col(teamB) > pl.col(teamA)
    res = res.with_columns(
        result=pl.when(teamAWins).then(pl.lit(teamA)).when(teamBWins).then(pl.lit(teamB)).otherwise(pl.lit("draw")).alias("result")
    )
    resSummedUp = res.group_by("result").agg(pl.col("probability").sum())
    res = res.join(resSummedUp, on="result", how="left").rename({"probability_right": "result_probability"})
    res = res.with_columns(
        expectedGain = (3*pl.col("probability") + pl.col("result_probability") - pl.col("probability"))/100)
    return res

In [ ]:
teamA = "South Korea"
teamB = "Czech Republic"
probScore = getStatsDf(teamA, teamB, 10, "neutral")
probScore = probScore.sort("expectedGain", descending=True)
probScore

In [ ]:
teamA = "Canada"
teamB = "Czech Republic"
probScore = getStatsDf(teamA, teamB, 10, "first")
probScore = probScore.sort("expectedGain", descending=True)
probScore

In [ ]:
teamA = "United States"
teamB = "Paraguay"
probScore = getStatsDf(teamA, teamB, 10, "first")
probScore = probScore.sort("expectedGain", descending=True)
probScore

In [ ]:
teamA = "Switzerland"
teamB = "Qatar"
probScore = getStatsDf(teamA, teamB, 10, "neutral")
probScore = probScore.sort("expectedGain", descending=True)
probScore

In [ ]:
teamAWins = probScore.filter(pl.col(teamA) > pl.col(teamB)).select(pl.col("probability").sum())[0,0]
teamBWins = probScore.filter(pl.col(teamB) > pl.col(teamA)).select(pl.col("probability").sum())[0,0]
draws = probScore.filter(pl.col(teamA) == pl.col(teamB)).select(pl.col("probability").sum())[0,0]
print(f"{teamA} wins: {teamAWins:.2f}%")
print(f"{teamB} wins: {teamBWins:.2f}%")
print(f"Draw: {draws:.2f}%")

In [ ]:
teamBWins

In [ ]:
dictAttack["South Africa"]

In [ ]:
dictDefend["Mexico"]

In [ ]:
dictDefend["South Africa"]

In [ ]:
df = pl.DataFrame({
    "team": allTeams,
    "attack": attack,
    "defend": defend
})

In [ ]:
df

In [ ]:
defend

In [ ]:
attack